# Find worst performing classes by model prediction

In [ ]:
from typing import List
from pydantic import BaseModel
import json
from pathlib import Path
#locals
from configs import get_class_list
from video_dataset import get_wlasl_info, get_labels_path, load_data_from_json
from utils import plt_display_grid, load_rgb_frames_from_video
from video_dataset import get_video_path, get_transform
from preprocess import WLASLClass, Instance
from stats import (
    AVAIL_SETS,
    AVAIL_SPLITS,
    get_per_instance_stats,
    plot_distribution,
    sort_distribution,
    HistoGram,
    reverse_preproc_format,
    to_preproc_format,
)
from run_types import MinInfo, AdminInfo
from models import avail_models, get_model
from configs import load_config
from testing import setup_data, test_topk_clsrep
import torch

Set some parameters

In [ ]:
verbosity = 1
save_files = False
def printv(*args, level=1, **kwargs):
    if level <= verbosity:
        print(*args, **kwargs)
        
split_idx = 3 #change for different split
split_options: List[AVAIL_SPLITS] = ["asl100", "asl300", "asl1000", "asl2000"]
set_options: List[AVAIL_SETS] = ['train', 'test', 'val']
split_name: AVAIL_SPLITS = split_options[split_idx]
set_idx = 1 #change for different set
set_name: AVAIL_SETS = set_options[set_idx]
classes = get_class_list()
check_name = 'best.pth'
printv(f'Split name: {split_name}')
printv(f'Set name: {set_name}')

Split name: asl2000
Set name: test


In [17]:
admin = AdminInfo.model_validate({
    "model" : "MViTv2_B_32x3",
    "dataset": "WLASL",
    "split": "asl2000",
    "save_path": "runs/asl2000/MViTv2_B_32x3/exp004/checkpoints",
    "exp_no": "004",
    "recover": False,
    "config_path": "configfiles/asl2000/MViTv2_B_32x3/exp004.toml",
    "weight_path": "runs/asl1000/MViTv2_B_32x3/exp001/checkpoints/best.pth"
})
config = load_config(admin)
save_path = Path(admin.save_path)



In [18]:
test_loader, num_classes, _, _ = setup_data(set_name, split_name, config.data)
print(f'Num classed: {num_classes}')
print(f'Num instances: {len(test_loader)}')

Num classed: 2000
Num instances: 2879


In [19]:
model = get_model(admin.model, num_classes, 0.0)
check_path = save_path / check_name
print(f"Loading weights from: {check_path}")
checkpoint = torch.load(check_path)
model.load_state_dict(checkpoint)


Loading weights from: runs/asl2000/MViTv2_B_32x3/exp004/checkpoints/best.pth


/tmp/ipykernel_2119962/2116541114.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(check_path)


<All keys matched successfully>

In [ ]:
print(f"Testing on {set_name} set")
topk_res, cls_report, all_targets, all_preds = test_topk_clsrep(
        model=model,
        test_loader=test_loader,
        verbose=False,
    )